### PASO 6 (STREAMLIT): APLICACIÓN WEB — SOLUCIÓN SUDOKU

### ¿Qué hace este notebook?

Crea la aplicación web con Streamlit que será la demo del proyecto.

El usuario sube una foto de un sudoku y la app:
1. Detecta la cuadrícula con YOLO
2. Corrige la perspectiva (warp)
3. Reconoce los dígitos con la CNN
4. Resuelve el sudoku con backtracking
5. Muestra la imagen original y la solución lado a lado

### Plan

```
CELDA 1 → Instalación de dependencias
CELDA 2 → Crear app.py (la aplicación completa)
CELDA 3 → Crear requirements.txt
CELDA 4 → Verificar archivos guardados
CELDA 5 → Instrucciones para ejecutar la app en local
```

#### CELDA 1 — Instalación

In [ ]:
# ============================================
# INSTALACIÓN
# ============================================

from google.colab import drive
drive.mount('/content/drive')

!pip install streamlit ultralytics tensorflow opencv-python-headless -q

import os
import streamlit

RUTA_SUDOKU = '/content/drive/MyDrive/sudoku/'
RUTA_APP    = RUTA_SUDOKU + 'app/'
os.makedirs(RUTA_APP, exist_ok=True)

print(f"✅ Streamlit {streamlit.__version__} instalado")
print(f"📁 Carpeta app: {RUTA_APP}")

#### CELDA 2 — Crear app.py

La aplicación completa en un solo archivo.
Importa el pipeline desde `pipeline.py` (generado en el Paso 4).

In [ ]:
# ============================================
# CREAR app.py
# ============================================

RUTA_SUDOKU = '/content/drive/MyDrive/sudoku/'
RUTA_APP    = RUTA_SUDOKU + 'app/'

app_py = '''"""
app.py — Sudoku Solver con Streamlit
Sube una foto de un sudoku y obtén la solución.

Para ejecutar:
    streamlit run app.py
"""

import streamlit as st
import cv2
import numpy as np
from PIL import Image
import tempfile
import os
import sys

# ── Configuración de la página ───────────────────────────────────────────
st.set_page_config(
    page_title="Sudoku Solver",
    page_icon="🧩",
    layout="wide"
)

# ── Rutas de los modelos ─────────────────────────────────────────────────
# Ajusta estas rutas si ejecutas en local
DIR_APP   = os.path.dirname(os.path.abspath(__file__))
RUTA_YOLO = os.path.join(DIR_APP, "modelo_yolo.pt")
RUTA_CNN  = os.path.join(DIR_APP, "modelo_digitos_v3.keras")

# ── Cargar modelos (solo una vez gracias a cache) ────────────────────────
@st.cache_resource
def cargar_modelos():
    """
    @st.cache_resource guarda los modelos en memoria entre ejecuciones.
    Sin esto, los modelos se recargarían cada vez que el usuario sube una foto.
    """
    sys.path.append(DIR_APP)
    from pipeline import cargar_modelos as _cargar
    return _cargar(RUTA_YOLO, RUTA_CNN)

# ── Interfaz principal ───────────────────────────────────────────────────
st.title("🧩 Sudoku Solver")
st.markdown("Sube una foto de un sudoku y lo resolveremos automáticamente.")
st.markdown("---")

# Comprobar que los modelos existen antes de cargar
if not os.path.exists(RUTA_YOLO) or not os.path.exists(RUTA_CNN):
    st.error("❌ No se encuentran los modelos. "
             "Asegúrate de que modelo_yolo.pt y modelo_digitos_v3.keras "
             "están en la misma carpeta que app.py.")
    st.stop()

# Cargar modelos
with st.spinner("Cargando modelos... (solo la primera vez)"):
    modelo_yolo, modelo_cnn = cargar_modelos()

# ── Subir imagen ─────────────────────────────────────────────────────────
foto = st.file_uploader(
    "Sube una foto del sudoku",
    type=["jpg", "jpeg", "png"],
    help="La foto debe mostrar la cuadrícula completa y estar bien iluminada."
)

if foto is not None:
    # Guardar la imagen subida en un archivo temporal
    with tempfile.NamedTemporaryFile(delete=False, suffix=".jpg") as tmp:
        tmp.write(foto.read())
        ruta_tmp = tmp.name

    # Mostrar imagen original
    img_original = Image.open(ruta_tmp)

    col1, col2 = st.columns(2)
    with col1:
        st.subheader("Foto original")
        st.image(img_original, use_column_width=True)

    # Procesar
    with st.spinner("Analizando el sudoku..."):
        sys.path.append(DIR_APP)
        from pipeline import mostrar_resultado

        overlay, matriz, solucion, mensaje = mostrar_resultado(
            ruta_tmp, modelo_yolo, modelo_cnn
        )

    # Mostrar resultado
    with col2:
        st.subheader("Solución")

        if overlay is not None:
            # Convertir BGR→RGB para Streamlit
            overlay_rgb = cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB) if len(overlay.shape) == 3 else overlay
            st.image(overlay_rgb, use_column_width=True)
            st.success(mensaje)
        else:
            st.image(img_original, use_column_width=True)
            st.error(mensaje)

    # ── Mostrar matrices ──────────────────────────────────────────────────
    if matriz is not None and solucion is not None:
        st.markdown("---")
        col3, col4 = st.columns(2)

        with col3:
            st.subheader("Dígitos detectados")
            tabla_detectada = ""
            for i, fila in enumerate(matriz):
                if i > 0 and i % 3 == 0:
                    tabla_detectada += "\n"
                fila_str = "  ".join(str(int(n)) if n != 0 else "·" for n in fila)
                tabla_detectada += fila_str + "\n"
            st.code(tabla_detectada)

        with col4:
            st.subheader("Solución completa")
            tabla_solucion = ""
            for i, fila in enumerate(solucion):
                if i > 0 and i % 3 == 0:
                    tabla_solucion += "\n"
                fila_str = "  ".join(str(int(n)) for n in fila)
                tabla_solucion += fila_str + "\n"
            st.code(tabla_solucion)

    # Limpiar archivo temporal
    os.unlink(ruta_tmp)

# ── Footer ────────────────────────────────────────────────────────────────
st.markdown("---")
st.markdown(
    "Proyecto Deep Learning · YOLO + CNN + Backtracking · Junio 2025",
    help="Stack: Ultralytics YOLO, TensorFlow/Keras, OpenCV, Streamlit"
)
'''

with open(RUTA_APP + 'app.py', 'w', encoding='utf-8') as f:
    f.write(app_py)

print(f"✅ app.py guardado en: {RUTA_APP}")
print(f"   Tamaño: {os.path.getsize(RUTA_APP + 'app.py') / 1024:.1f} KB")

#### CELDA 3 — Crear requirements.txt

In [ ]:
# ============================================
# CREAR requirements.txt
# ============================================

requirements = """streamlit>=1.28.0
ultralytics>=8.0.0
tensorflow>=2.13.0
opencv-python-headless>=4.8.0
numpy>=1.24.0
Pillow>=9.5.0
"""

with open(RUTA_APP + 'requirements.txt', 'w') as f:
    f.write(requirements)

print(f"✅ requirements.txt guardado")
print()
print("Contenido:")
print(requirements)

#### CELDA 4 — Verificar archivos guardados

In [ ]:
# ============================================
# VERIFICAR ARCHIVOS
# ============================================

print("📁 Contenido de la carpeta app/:")
print("="*50)

archivos_necesarios = [
    'app.py',
    'pipeline.py',          # generado en el Paso 4
    'requirements.txt',
    'modelo_yolo.pt',       # debes copiarlo aquí
    'modelo_digitos_v3.keras'  # debes copiarlo aquí
]

for archivo in archivos_necesarios:
    ruta = RUTA_APP + archivo
    if os.path.exists(ruta):
        tamaño = os.path.getsize(ruta) / 1024
        print(f"   ✅ {archivo:<30} {tamaño:>8.1f} KB")
    else:
        print(f"   ❌ {archivo:<30} NO ENCONTRADO")

# Copiar modelos a la carpeta app/ si no están
for modelo, origen in [
    ('modelo_yolo.pt',          RUTA_SUDOKU + 'modelo_yolo.pt'),
    ('modelo_digitos_v3.keras', RUTA_SUDOKU + 'modelo_digitos_v3.keras')
]:
    destino = RUTA_APP + modelo
    if not os.path.exists(destino) and os.path.exists(origen):
        import shutil
        shutil.copy2(origen, destino)
        print(f"\n   📋 {modelo} copiado a app/")

#### CELDA 5 — Instrucciones para ejecutar

Cómo lanzar la app en local.

In [ ]:
# ============================================
# INSTRUCCIONES DE EJECUCIÓN
# ============================================

print("""
╔══════════════════════════════════════════════════════════════╗
║           INSTRUCCIONES PARA EJECUTAR LA APP                ║
╠══════════════════════════════════════════════════════════════╣
║  1. Descarga la carpeta app/ de Google Drive                ║
║  2. Abre una terminal en esa carpeta                        ║
║  3. Instala dependencias:                                    ║
║       pip install -r requirements.txt                        ║
║  4. Ejecuta:                                                 ║
║       streamlit run app.py                                   ║
║  5. Se abrirá en http://localhost:8501                      ║
║                                                              ║
║
║  ESTRUCTURA NECESARIA EN LA CARPETA app/:                   ║
║  ─────────────────────────────────────────────────────────  ║
║  app/                                                        ║
║  ├── app.py                    ← aplicación principal       ║
║  ├── pipeline.py               ← funciones del pipeline     ║
║  ├── requirements.txt          ← dependencias               ║
║  ├── modelo_yolo.pt            ← modelo YOLO                ║
║  └── modelo_digitos_v3.keras   ← modelo CNN                 ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")